In [ ]:
import pyspark.sql.functions as F
from pyspark.sql import Window
from delta.tables import DeltaTable

In [ ]:
silver_path     = "tihim_project.silver.orders"
dim_customers   = "tihim_project.gold.dim_customers"
dim_products    = "tihim_project.gold.dim_products"
gold_path       = "tihim_project.gold.fact_orders"
checkpoint_path = "/Volumes/tihim_project/ops/stream_state/checkpoints/gold/orders"

In [ ]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {gold_path} (
        order_sk       STRING,   
        order_id       STRING,    
        customer_sk    STRING, 
        product_sk     STRING,  
        order_date     DATE,     
        order_date_key INT,      
        quantity       INT,       
        unit_price     DOUBLE,  
        total_amount   DOUBLE,    
        etl_updated_at TIMESTAMP  
    )
    USING DELTA
""")

In [ ]:
silver_stream = spark.readStream.option("readChangeFeed", "true").table(silver_path)

def prep_changes(df):
    df = df.filter(F.col("_change_type").isin("insert", "update_postimage"))
    w = Window.partitionBy("order_id").orderBy(F.col("_commit_version").desc())
    return df.withColumn("rn", F.row_number().over(w)).filter(F.col("rn") == 1).drop("rn")

In [ ]:
def upsert_orders_to_fact(microbatch_df, batch_id):
    changes = prep_changes(microbatch_df)
    if changes.isEmpty():
        return

    cust = (spark.table(dim_customers).filter(F.col("is_active") == True)
                 .select("customer_id", "customer_sk"))
    prod = (spark.table(dim_products).filter(F.col("is_active") == True)
                 .select("product_id", "product_sk"))

    fact = (changes.alias("o")
        .join(cust, "customer_id", "left")
        .join(prod, "product_id", "left")

        .withColumn("customer_sk", F.coalesce(F.col("customer_sk"), F.lit("-1")))
        .withColumn("product_sk",  F.coalesce(F.col("product_sk"),  F.lit("-1")))
        .withColumn("order_date_key", F.date_format(F.col("order_date"), "yyyyMMdd").cast("int"))
        .withColumn("order_sk", F.substring(F.sha2(F.col("order_id"), 256), 1, 16))
        .withColumn("etl_updated_at", F.current_timestamp())
        .select(
            "order_sk", 
            "order_id", 
            "customer_sk", 
            "product_sk",
            "order_date", 
            "order_date_key", 
            "quantity", 
            "unit_price",
            "total_amount", 
            "etl_updated_at"))

    try:
        (DeltaTable.forName(spark, gold_path).alias("target").merge(
            
            source = fact.alias("update"),
            condition = "target.order_id = update.order_id"
        ).whenMatchedUpdate(
            condition = """
                NOT (target.customer_sk  <=> update.customer_sk)  OR
                NOT (target.product_sk   <=> update.product_sk)   OR
                NOT (target.quantity     <=> update.quantity)     OR
                NOT (target.unit_price   <=> update.unit_price)   OR
                NOT (target.total_amount <=> update.total_amount) OR
                NOT (target.order_date   <=> update.order_date)
            """,
            set = {
                "customer_sk": "update.customer_sk", 
                "product_sk": "update.product_sk",
                "order_date": "update.order_date", 
                "order_date_key": "update.order_date_key",
                "quantity": "update.quantity", 
                "unit_price": "update.unit_price",
                "total_amount": "update.total_amount", 
                "etl_updated_at": "update.etl_updated_at"
            }
        ).whenNotMatchedInsert(
            values = {
                "order_sk": "update.order_sk", 
                "order_id": "update.order_id",
                "customer_sk": "update.customer_sk", 
                "product_sk": "update.product_sk",
                "order_date": "update.order_date", 
                "order_date_key": "update.order_date_key",
                "quantity": "update.quantity", 
                "unit_price": "update.unit_price",
                "total_amount": "update.total_amount", 
                "etl_updated_at": "update.etl_updated_at"
            }
        ).execute())
    
    except Exception as e:
        print(f"[fact_orders] batch {batch_id} failed: {e}")
        raise

In [ ]:
query = (silver_stream.writeStream.foreachBatch(upsert_orders_to_fact)
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True).start()
    )
    
query.awaitTermination()